# MIMIC Sepsis Shock Prediction with Trajectory Features
Feature configurations: baseline, summary stats, trajectory probs, and combinations.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import os
import sys
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))
from notebook_utils import biomarker_summary_stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports successful')

## Load Data

In [ ]:
pred_with_probs_path = '../../../results/mimic/sepsis/sepsis_prediction_dataset_with_probs.csv'
outcome_path = '../../../results/mimic/sepsis/septic_shock_outcomes.csv'
lactate_path = '../../../results/mimic/sepsis/lactate_timeseries.csv'
wbc_path = '../../../results/mimic/sepsis/wbc_timeseries.csv'
platelet_path = '../../../results/mimic/sepsis/platelet_timeseries.csv'

df = pd.read_csv(pred_with_probs_path)
outcomes = pd.read_csv(outcome_path)
lactate_ts = pd.read_csv(lactate_path)
wbc_ts = pd.read_csv(wbc_path)
platelet_ts = pd.read_csv(platelet_path)

print(f'✓ Loaded prediction dataset with probs: {len(df):,} rows')
print(f'✓ Outcomes: {len(outcomes):,} rows')
print(f'✓ Lactate TS: {len(lactate_ts):,} rows')
print(f'✓ WBC TS: {len(wbc_ts):,} rows')
print(f'✓ Platelet TS: {len(platelet_ts):,} rows')

df = df.merge(outcomes[['hadm_id', 'time_day', 'target_septic_shock']], on=['hadm_id', 'time_day'], how='left')
df = df[df['target_septic_shock'].notna()].copy()
df['target_septic_shock'] = df['target_septic_shock'].astype(int)

traj_cols = [c for c in df.columns if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]

print(f'Final dataset rows: {len(df):,}')
print(f'Outcome rate: {df["target_septic_shock"].mean():.1%}')
print(f'Trajectory columns: {len(traj_cols)}')

## Feature Engineering

In [ ]:
lactate_3d = biomarker_summary_stats(lactate_ts, 'lactate', lookback_days=3)
wbc_3d = biomarker_summary_stats(wbc_ts, 'wbc', lookback_days=3)
platelet_3d = biomarker_summary_stats(platelet_ts, 'platelet', lookback_days=3)

df = df.merge(lactate_3d, on=['hadm_id', 'time_day'], how='left')
df = df.merge(wbc_3d, on=['hadm_id', 'time_day'], how='left')
df = df.merge(platelet_3d, on=['hadm_id', 'time_day'], how='left')

exclude_cols = {'hadm_id', 'time_day', 'charttime', 'admittime', 'dischtime', 'target_septic_shock'}
numeric_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
traj_cols = [c for c in numeric_cols if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]
summary_cols = [c for c in numeric_cols if c.endswith('_3d') or '_trend_' in c or '_change_' in c]
base_cols = [c for c in numeric_cols if c not in traj_cols and c not in summary_cols]

static_name_tokens = ['age', 'gender', 'sex', 'baseline', 'admit', 'admission', 'ethnicity', 'race', 'height', 'weight', 'bmi']
static_cols = [c for c in base_cols if any(tok in c.lower() for tok in static_name_tokens)]
dynamic_cols = [c for c in base_cols if c not in static_cols]

feature_sets = {
    'Trajectory Only': traj_cols,
    'Summary Stats Only': summary_cols,
    'Trajectory + Summary Stats': traj_cols + summary_cols,
    'Static Only': static_cols,
    'Trajectory + Static': traj_cols + static_cols,
    'Summary Stats + Static': summary_cols + static_cols,
    'Trajectory + Summary Stats + Static': traj_cols + summary_cols + static_cols,
}

if len(dynamic_cols) > 0:
    feature_sets['Static + Dynamic'] = static_cols + dynamic_cols
    feature_sets['Trajectory + Static + Dynamic'] = traj_cols + static_cols + dynamic_cols
    feature_sets['Summary + Static + Dynamic'] = summary_cols + static_cols + dynamic_cols
    feature_sets['Trajectory + Summary + Static + Dynamic'] = traj_cols + summary_cols + static_cols + dynamic_cols

for name, cols in feature_sets.items():
    print(f'{name}: {len(cols)} features')

## Model Comparison

In [ ]:
models_to_evaluate = {
    'LogReg': lambda: LogisticRegression(max_iter=300, n_jobs=-1),
    'RF': lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    'HGB': lambda: HistGradientBoostingClassifier(random_state=42),
    'XGB': lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, eval_metric='logloss'),
}

n_repeats = 10
n_splits = 5
results = []
y = df['target_septic_shock']
groups = df['hadm_id']

for model_name, model_fn in models_to_evaluate.items():
    dataset_model = df.copy()
    if len(traj_cols) > 0:
        dataset_model[traj_cols] = dataset_model.groupby('hadm_id')[traj_cols].ffill(limit=2)
    if model_name not in ['XGB', 'HGB']:
        if len(traj_cols) > 0:
            dataset_model[traj_cols] = dataset_model[traj_cols].fillna(0)
        if len(summary_cols) > 0:
            dataset_model[summary_cols] = dataset_model[summary_cols].fillna(0)

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            continue
        aucs, auprcs = [], []
        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920 + repeat).permutation(len(dataset_model))
            dataset_repeat = dataset_model.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_splits)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                if model_name not in ['XGB', 'HGB']:
                    imputer = SimpleImputer(strategy='median')
                    X_train_imputed = imputer.fit_transform(X_train)
                    X_test_imputed = imputer.transform(X_test)
                else:
                    X_train_imputed = X_train
                    X_test_imputed = X_test

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                model = model_fn()
                model.fit(X_train_scaled, y_train)

                if hasattr(model, 'predict_proba'):
                    probs = model.predict_proba(X_test_scaled)[:, 1]
                else:
                    probs = model.decision_function(X_test_scaled)

                aucs.append(roc_auc_score(y_test, probs))
                auprcs.append(average_precision_score(y_test, probs))

        print(f"{model_name} | {feature_set_name}: AUROC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}, AUPRC {np.mean(auprcs):.3f} ± {np.std(auprcs):.3f}")
        results.append({
            'model': model_name,
            'feature_set': feature_set_name,
            'auroc_mean': np.mean(aucs),
            'auroc_std': np.std(aucs),
            'auprc_mean': np.mean(auprcs),
            'auprc_std': np.std(auprcs),
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:
summary = results_df.pivot_table(index='feature_set', columns='model', values='auroc_mean')
summary

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='feature_set', y='auroc_mean', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUROC by Feature Set and Model')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=results_df, x='feature_set', y='auprc_mean', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUPRC by Feature Set and Model')
plt.tight_layout()
plt.show()